# 01b · CuratorKIT: safety-focused curation of the Alpaca fine-tuning data

General instruction tuning on Alpaca is the benign condition that eroded safety (08). This notebook runs `ft_alpaca.jsonl` through Lexsi Labs' **CuratorKIT** (MinHash deduplication, cleaning, schema gate, secrets gate and the Detoxify toxicity gate; no LLM) and writes `ft_alpacacur.jsonl` for fine-tuning run `alpacacur-s0` (07), which 08 then compares with `alpaca-s0`: does curated data drift less?

| | |
|---|---|
| **Kaggle settings** | Accelerator: None (CPU) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · output of **01** (or `sp-pipeline-outputs`) |
| **Reads** | `ft_alpaca.jsonl` |
| **Writes** (`/kaggle/working/`) | `data/ft_alpacacur.jsonl`, `data/curation_report.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

/root/.cache/uv/environments-v2/job3-curatorkit-dca255f993ff2c3c/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'torch': '2.15.0+cu132',
 'transformers': '5.22.2',
 'cuda_available': True,
 'on_kaggle': False,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['NVIDIA A100-SXM4-80GB'],
 'capability': [8, 0],
 'vram_gb': 85.1}

In [2]:
import json, subprocess, sys
from collections import Counter

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "curatorkit==0.1.22", "detoxify>=0.5",
                "detect-secrets>=1.4"], check=True)
from curatorkit import Curator, CuratorConfig

OUT = sp.out_dir("data")
rows = sp.load_jsonl(sp.find_input("ft_alpaca.jsonl"))
# CuratorKIT reads Alpaca-format SFT data (instruction / input / output); ours are 2-turn chat messages.
src = OUT / "_alpaca_for_curator.jsonl"
sp.save_jsonl([{"id": r["id"], "instruction": r["messages"][0]["content"], "input": "",
                "output": r["messages"][1]["content"]} for r in rows], src)
print(len(rows), "Alpaca training examples")

2000 Alpaca training examples


In [3]:
# Only the safety- and hygiene-related filters are on. Length limits are opened up so that any change in
# drift cannot come from simply dropping short or long examples.
cfg = CuratorConfig(dataset=str(src), format="alpaca", name="safetypulse_alpaca_safety",
                    output_dir=str(OUT / "curator"), export_formats=["alpaca"],
                    min_tokens=1, max_tokens=100_000, clean=True, dedup="minhash", schema_gate=True,
                    secrets_gate=True, toxicity_gate=True, toxicity_llm_judge=False)
result = Curator(cfg).run()
print(result.summary())

SchemaGate:   0%|          | 0/2000 [00:00<?, ?sample/s]

SchemaGate: 100%|██████████| 2000/2000 [00:00<00:00, 74963.21sample/s]

ExactDeduplicator:   0%|          | 0/1998 [00:00<?, ?sample/s]

ExactDeduplicator: 100%|██████████| 1998/1998 [00:00<00:00, 115374.40sample/s]

MinHashDeduplicator:   0%|          | 0/1998 [00:00<?, ?sample/s]

MinHashDeduplicator:   4%|▍         | 76/1998 [00:01<00:25, 74.56sample/s]

... (436 outputs left out)

ToxicityGate: 100%|█████████▉| 1993/1998 [07:13<00:01,  4.23sample/s]

ToxicityGate: 100%|██████████| 1998/1998 [07:14<00:00,  4.43sample/s]

ToxicityGate: 100%|██████████| 1998/1998 [07:14<00:00,  4.60sample/s]

────────────────────────────────────────────
  passed   :    1,981
  rejected :       19
  time     :   475.6s
  output   : /root/.cache/huggingface/hub/datasets--purvansh01--safetypulse-jobs/snapshots/0ab976f73b8ba0b09f2328571bfb81a651baa197/work/data/curator
────────────────────────────────────────────


In [4]:
curated = sp.load_jsonl(OUT / "curator" / "sft_alpaca.jsonl")
ft = [{"id": f"fc-{i:05d}", "messages": [{"role": "user", "content": (c["instruction"] + ("\n\n" + c["input"] if c.get("input") else "")).strip()},
                                         {"role": "assistant", "content": c["output"]}]} for i, c in enumerate(curated)]
sp.save_jsonl(ft, OUT / "ft_alpacacur.jsonl")
# rejection_reason is structured ("toxicity:…", "near_duplicate:…"): keep the category, never sample text.
reasons = Counter(str(getattr(r, "rejection_reason", "unknown")).split(":")[0] for r in result.rejected)
report = {"curatorkit": "0.1.22", "n_in": len(rows), "n_out": len(ft), "stage_counts": result.stage_counts,
          "rejected_reasons": dict(reasons.most_common()), "seconds": result.wall_clock_seconds,
          "config": {"dedup": "minhash", "clean": True, "schema_gate": True, "secrets_gate": True,
                     "toxicity_gate": "detoxify unbiased, pass < 0.1, reject >= 0.5, borderline rejected",
                     "min_tokens": 1}}
sp.save_json(report, OUT / "curation_report.json")
src.unlink()
{k: report[k] for k in ("n_in", "n_out", "rejected_reasons")}

{'n_in': 2000,
 'n_out': 1981,
 'rejected_reasons': {'toxic_content': 17, 'missing_field': 2}}